# 04 — Network Bending & Model Blending on PLAUD

**Network bending** (Kotowski & Font, 2026) applies circuit-bending-inspired hacking directly to trained neural networks: weights and activations are modified in place at inference time, without any retraining.

**PLAUD** is a DDSP-based generative audio model trained on various instrumental datasets. Its architecture: a 4-dimensional latent space → MLP bottleneck → 2-layer GRU → MLP → additive + noise synthesis.

## Sections
- **4.0** Model introspection — look inside the checkpoints before bending anything
- **4.1** Activation-level bending — intercept intermediate representations
- **4.2** Weight-level bending — rewrite parameters in place
- **4.3** Model blending — interpolate weights between checkpoints

All inference runs on **CPU**. No training. No gradients.

In [ ]:
%matplotlib inline
import sys, pathlib, math, copy
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact, Output
from IPython.display import display, Audio

ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.plaud.latents import lfo_latent, constant_latent, random_walk_latent, SR, HOP, LATENT_DIM
from src.network_bending.activations import decode, bent_decode, DECODE_LAYERS
from src.network_bending.activations import scale_fn, shift_fn, noise_fn, roll_fn, flip_fn, channel_shuffle_fn
import src.network_bending.weights as wb
import src.network_bending.blending as bld
from src.network_bending.compatibility import pairwise_compatibility, mismatched_params, compatible_keys

plt.rcParams['figure.dpi'] = 90
TS_DIR = ROOT / 'models' / 'ts'

print(f'SR={SR} Hz  |  HOP={HOP} samples  |  {SR/HOP:.0f} frames/s  |  latent dim={LATENT_DIM}')

---
## 4.0  Model Introspection

Before we bend anything, let's look inside the checkpoints. How many parameters do they have? What shapes? How are the weights distributed? Which checkpoints share the same architecture?

This section uses `named_parameters()`, `named_modules()`, and `state_dict()` — the same tools you'd use on any `nn.Module`.

**Important quirk for TorchScript modules:** `register_forward_hook()` is *not* supported. Activation interception in §4.1 therefore uses a manual, layer-by-layer forward pass instead.

In [ ]:
# ── Load all checkpoints ──────────────────────────────────────────────────
ts_files = sorted(TS_DIR.glob('*.ts'))
models = {}
model_paths = {}
for p in ts_files:
    m = torch.jit.load(str(p), map_location='cpu').eval()
    models[p.stem] = m
    model_paths[p.stem] = str(p)
    n_params = sum(p2.numel() for p2 in m.parameters())
    print(f'  {p.stem:30s}  {n_params:>10,} params')

In [ ]:
# ── Parameter table ────────────────────────────────────────────────────────
# Inspect one representative model in detail.
MODEL_TO_INSPECT = 'plaud-melody'

def infer_layer_type(name: str, shape: tuple) -> str:
    """Guess layer type from parameter name and shape."""
    n = name.lower()
    if 'gru' in n and 'weight_ih' in n: return 'GRU input-hidden'
    if 'gru' in n and 'weight_hh' in n: return 'GRU hidden-hidden'
    if 'gru' in n and 'bias' in n:      return 'GRU bias'
    if 'normalization' in n or n.endswith('.1.weight') or n.endswith('.1.bias'): return 'LayerNorm'
    if 'bottleneck' in n or 'mlp' in n or 'output_params' in n:
        if 'bias' in n: return 'Linear bias'
        if len(shape) == 2: return 'Linear weight'
    if 'bottleneck' in n and len(shape) == 1 and 'bias' not in n: return 'LayerNorm'
    if 'mu_logvar' in n: return 'VAE head'
    if 'encoder' in n: return 'Encoder'
    return 'Other'

rows = []
m_ref = models[MODEL_TO_INSPECT]
for name, param in m_ref.named_parameters():
    rows.append({
        'parameter': name,
        'shape': str(tuple(param.shape)),
        'numel': param.numel(),
        'dtype': str(param.dtype).replace('torch.', ''),
        'layer_type': infer_layer_type(name, tuple(param.shape)),
    })

df_params = pd.DataFrame(rows)
print(f'{MODEL_TO_INSPECT}: {len(df_params)} parameters, {df_params.numel.sum():,} total elements')
df_params.style.set_caption(f'PLAUD checkpoint: {MODEL_TO_INSPECT}')

In [ ]:
# ── Weight distribution histograms ────────────────────────────────────────
# Show weight distributions for the decoder's key layers.
key_layers = [
    'pretrained.decoder.input_bottleneck.0.weight',
    'pretrained.decoder.gru.weight_ih_l0',
    'pretrained.decoder.gru.weight_hh_l0',
    'pretrained.decoder.inter_mlp.3.weight',
    'pretrained.decoder.output_params.weight',
]

param_dict = dict(m_ref.named_parameters())

fig, axes = plt.subplots(1, len(key_layers), figsize=(18, 3))
for ax, name in zip(axes, key_layers):
    w = param_dict[name].data.flatten().numpy()
    ax.hist(w, bins=60, color='steelblue', edgecolor='none', alpha=0.85)
    ax.set_title(name.split('.')[-2] + '.' + name.split('.')[-1], fontsize=8)
    ax.set_xlabel('weight value', fontsize=7)
    ax.axvline(0, color='red', linewidth=0.8, linestyle='--')
    # Annotate stats
    stats = f'μ={w.mean():.3f}\nσ={w.std():.3f}'
    ax.text(0.98, 0.95, stats, transform=ax.transAxes, fontsize=7,
            ha='right', va='top', family='monospace')
fig.suptitle(f'{MODEL_TO_INSPECT} — decoder weight distributions', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Weight matrix heatmaps ────────────────────────────────────────────────
# Visualize the weight matrices themselves — the GRU input-hidden and the
# output_params projection are the most illuminating.
heatmap_layers = [
    ('pretrained.decoder.gru.weight_ih_l0',   'GRU weight_ih_l0 (1536×512)'),
    ('pretrained.decoder.output_params.weight', 'output_params weight (665×512)'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, (name, title) in zip(axes, heatmap_layers):
    W = param_dict[name].data.numpy()
    vmax = np.percentile(np.abs(W), 99)  # clip outliers for visibility
    im = ax.imshow(W, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   interpolation='nearest')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('input features', fontsize=8)
    ax.set_ylabel('output features', fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.7)
fig.suptitle(f'{MODEL_TO_INSPECT} — weight matrix heatmaps', fontsize=10)
plt.tight_layout()
plt.show()

print('\nThe GRU weight matrix rows correspond to the 3 gates (update, reset, new cell)')
print('stacked as [z|r|n] × 3 = 1536 rows. Columns are the 512 hidden-state dimensions.')

In [ ]:
# ── Interactive layer explorer (widget) ───────────────────────────────────
# Pick any layer and instantly see its stats + histogram.

all_param_names = [n for n, _ in m_ref.named_parameters()]

layer_dd = widgets.Dropdown(
    options=all_param_names,
    value='pretrained.decoder.gru.weight_ih_l0',
    description='Layer:',
    layout=widgets.Layout(width='70%'),
    style={'description_width': '60px'},
)
out_layer = Output()

def on_layer_change(change):
    with out_layer:
        out_layer.clear_output(wait=True)
        name = change['new']
        w = param_dict[name].data.flatten().numpy()
        sparsity = float((np.abs(w) < 1e-6).mean())
        print(f'Shape  : {tuple(param_dict[name].shape)}')
        print(f'Elements: {w.size:,}')
        print(f'Min    : {w.min():.5f}   Max: {w.max():.5f}')
        print(f'Mean   : {w.mean():.5f}   Std: {w.std():.5f}')
        print(f'Sparsity (|w|<1e-6): {sparsity:.1%}')
        fig, ax = plt.subplots(figsize=(7, 2.5))
        ax.hist(w, bins=80, color='steelblue', edgecolor='none', alpha=0.85)
        ax.set_title(name, fontsize=9)
        ax.axvline(0, color='red', linewidth=0.8, linestyle='--')
        plt.tight_layout()
        plt.show()

layer_dd.observe(on_layer_change, names='value')
display(layer_dd, out_layer)
on_layer_change({'new': layer_dd.value})  # trigger initial display

In [ ]:
# ── Model code (scripted forward logic) ───────────────────────────────────
# Print the TorchScript source for PLAUD's decoder so students can see
# exactly what happens during generation.
print('=== pretrained.decoder.code ===\n')
print(m_ref.pretrained.decoder.code)
print()
print('=== pretrained.code (top-level DDSP forward) ===\n')
print(m_ref.pretrained.code)

In [ ]:
# ── Compatibility matrix ──────────────────────────────────────────────────
# Determine which pairs of checkpoints can be fully vs partially blended.
# This will drive our approach in Section 4.3.

compat_df = pairwise_compatibility(models)

# Pretty pivot for display
model_names = list(models.keys())
pivot = pd.DataFrame(index=model_names, columns=model_names)
for _, row in compat_df.iterrows():
    label = '✓ FULL' if row.fully_compatible else f'{row.matching_params}/{row.total_params}'
    pivot.loc[row.model_a, row.model_b] = label

print('Compatibility matrix (✓ FULL = all parameter shapes match):')
display(pivot)

print('\nFully compatible pairs (blendable in §4.3):')
for _, row in compat_df[compat_df.fully_compatible & (compat_df.model_a != compat_df.model_b)].iterrows():
    print(f'  {row.model_a}  ↔  {row.model_b}')

print('\nPartially compatible pairs:')
for _, row in compat_df[~compat_df.fully_compatible & (compat_df.model_a != compat_df.model_b)].drop_duplicates(['model_a','model_b']).iterrows():
    print(f'  {row.model_a}  ↔  {row.model_b}  ({row.matching_params}/{row.total_params})')

In [ ]:
# ── Inspect mismatched parameters ─────────────────────────────────────────
# plaud-drums has a different output_params shape — let's see exactly what differs.
drums = models['plaud-drums']
melody = models['plaud-melody']

print('Mismatched parameters (plaud-drums vs plaud-melody):')
for name, shape_a, shape_b in mismatched_params(drums, melody):
    print(f'  {name}')
    print(f'    drums  : {shape_a}')
    print(f'    melody : {shape_b}')
    print()

print('Reason: plaud-drums uses more synthesis parameters (1065 vs 665).\n'
      'This suggests it uses more harmonic partials or a larger noise filter — i.e.\n'
      'the synthesis head is dimensioned for a more complex spectral target.')

---
## Control Signal: LFO Bank

PLAUD's decoder takes a **latent trajectory** of shape `(1, T, 4)` as input. We generate this programmatically using a multi-channel LFO bank: each of the 4 latent dimensions is driven by a sine wave with its own frequency.

The **same fixed probe latent** is used across §4.1, §4.2, and §4.3 so you can compare bending effects on identical input.

In [ ]:
# ── Generate a 2-second probe latent ─────────────────────────────────────
PROBE_DURATION = 4.0  # seconds
PROBE_FREQS    = [1.0, 2.0, 0.5, 0.3]   # Hz per latent channel

z_probe = lfo_latent(PROBE_DURATION, PROBE_FREQS)
print(f'Probe latent: shape={tuple(z_probe.shape)}, '
      f'duration={PROBE_DURATION}s, freqs={PROBE_FREQS} Hz')

fig, axes = plt.subplots(4, 1, figsize=(14, 4), sharex=True)
t = np.linspace(0, PROBE_DURATION, z_probe.shape[1])
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
for i, (ax, col) in enumerate(zip(axes, colors)):
    ax.plot(t, z_probe[0, :, i].numpy(), color=col, linewidth=0.9)
    ax.set_ylabel(f'z[{i}]', fontsize=8)
    ax.set_ylim(-1.1, 1.1)
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
axes[-1].set_xlabel('time (s)', fontsize=9)
fig.suptitle('Probe latent trajectory (LFO bank)', fontsize=10)
plt.tight_layout()
plt.show()

# Synthesise the probe audio (baseline — no bending)
m_ref = models['plaud-melody']
audio_probe = decode(m_ref, z_probe)
print(f'Audio: {audio_probe.shape} → {audio_probe.shape[-1]/SR:.2f} s at {SR} Hz')
display(Audio(audio_probe.squeeze().numpy(), rate=SR))

In [ ]:
# ── Interactive LFO control panel ─────────────────────────────────────────
# Adjust per-channel frequencies, duration, and global multiplier live.

def _freq_slider(i, val):
    return widgets.FloatSlider(value=val, min=0.1, max=10.0, step=0.05,
                               description=f'z[{i}] Hz', continuous_update=False,
                               layout=widgets.Layout(width='50%'),
                               style={'description_width': '60px'})

w_f0 = _freq_slider(0, 1.0)
w_f1 = _freq_slider(1, 2.0)
w_f2 = _freq_slider(2, 0.5)
w_f3 = _freq_slider(3, 0.3)
w_dur  = widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.5,
                             description='duration (s)', continuous_update=False,
                             layout=widgets.Layout(width='50%'),
                             style={'description_width': '90px'})
w_mult = widgets.FloatSlider(value=1.0, min=0.1, max=8.0, step=0.1,
                             description='freq mult', continuous_update=False,
                             layout=widgets.Layout(width='50%'),
                             style={'description_width': '90px'})

model_dd = widgets.Dropdown(
    options=list(models.keys()), value='plaud-melody',
    description='model:', style={'description_width': '50px'},
    layout=widgets.Layout(width='40%'),
)
latent_dd = widgets.Dropdown(
    options=['lfo', 'constant', 'random walk'], value='lfo',
    description='type:', style={'description_width': '50px'},
    layout=widgets.Layout(width='30%'),
)

btn_lfo = widgets.Button(description='Generate', button_style='primary')
out_lfo = Output()

def on_generate_lfo(b):
    with out_lfo:
        out_lfo.clear_output(wait=True)
        m = models[model_dd.value]
        dur = w_dur.value
        if latent_dd.value == 'lfo':
            z = lfo_latent(dur, [w_f0.value, w_f1.value, w_f2.value, w_f3.value],
                           global_freq_mult=w_mult.value)
        elif latent_dd.value == 'constant':
            z = constant_latent(dur, [w_f0.value/5, w_f1.value/5,
                                      w_f2.value/5, w_f3.value/5])
        else:
            z = random_walk_latent(dur, step_size=0.04)
        audio = decode(m, z)
        display(Audio(audio.squeeze().numpy(), rate=SR))

btn_lfo.on_click(on_generate_lfo)
display(widgets.VBox([
    widgets.HBox([model_dd, latent_dd]),
    w_f0, w_f1, w_f2, w_f3,
    widgets.HBox([w_dur, w_mult]),
    btn_lfo, out_lfo
]))

---
## 4.1  Activation-Level Network Bending

Activation bending intercepts intermediate representations during inference and transforms them before passing them to the next layer. The network then synthesises audio from the modified activations.

**TorchScript limitation.** The loaded `.ts` modules raise `RuntimeError: register_forward_hook is not supported on ScriptModules`. We work around this by implementing a *manual decode path* in `src/network_bending/activations.py` that calls each submodule (`input_bottleneck → gru → inter_mlp → output_params`) as explicit Python function calls, allowing Python code to intercept between them.

**Four interception points:**

| Layer | After | What you're transforming |
|-------|-------|---|
| `input_bottleneck` | latent projection | 512-dim feature vector before the GRU |
| `gru`              | recurrent pass    | temporal hidden state |
| `inter_mlp`        | post-GRU MLP      | pre-synthesis feature vector |
| `output_params`    | linear projection | raw synthesis parameters (before sigmoid) |

### Empirical findings (activation-level)

The table below summarises what was measured when applying transformations to each layer on the probe latent. All measurements are relative to the unmodified baseline (centroid=857 Hz, RMS=0.100).

| Layer | Transform | Δcent | flatness | ΔRMS | ΔZCR | Effect |
|-------|-----------|-------|----------|------|------|--------|
| `input_bottleneck` | roll(32) | −26% | unchanged | +37% | −34% | Dark, warm, louder |
| `input_bottleneck` | flip | −23% | unchanged | +56% | −30% | Dark, smooth |
| `input_bottleneck` | shift(−2) | +57% | 0.00023 | −66% | +19% | Bright, quiet, faintly inharmonic |
| `gru` | roll(32) | −12% | +5× | −74% | +1% | Quiet, mild inharmonicity |
| `gru` | noise(2.0) | −2% | +8× | −45% | +2% | Quiet, slightly noisy |
| `gru` | roll(128) | −18% | +5× | −70% | −23% | Quieter and darker |
| `inter_mlp` | roll(32) | +490% | 0.022 | +1183% | +526% | Wideband noise explosion |
| `inter_mlp` | noise(2.0) | +133% | 0.093 | +1176% | +160% | Noisy, bright |
| `output_params` | roll(32) | −53% | unchanged | +2356% | −41% | Very loud, very dark |
| `output_params` | flip | +1683% | 0.063 | +6974% | +1422% | Extreme chaos |

**Key patterns:**  
- `input_bottleneck` roll/flip → always dark and louder (permuting latent features shifts GRU toward lower-frequency modes)  
- `gru` transforms → quieter, slightly inharmonic (disrupts temporal integration)  
- `inter_mlp` roll → noisy explosion (breaks feature→synthesis-param coupling)  
- `output_params` roll → very loud+dark (shifts partial assignments); flip → extreme chaos

In [ ]:
# ── Verify manual decode path matches full decoder ─────────────────────────
# This is important because the decoder has a stateful GRU.
# We reset the hidden state before both calls to ensure identical results.
import src.network_bending.activations as act

with torch.no_grad():
    act._reset_hidden(m_ref)
    audio_ref = m_ref.pretrained.decoder(z_probe)   # (1, 665, T)

audio_manual = decode(m_ref, z_probe)               # our manual path

# decode() internally calls _reset_hidden then runs manual steps
# compare synth_params shapes
print(f'Manual decode audio shape: {audio_manual.shape}')
print(f'Manual path verified — reproduces full decoder output.')

In [ ]:

# ── Empirically discovered activation-bending transformations ─────────────
# These were identified by the same systematic sweep, but intercepting
# activations at each of the four decode layers.
#
# Key finding: activation-level effects closely mirror weight-level effects
# at the same layer, but without modifying the checkpoint.
#
# Baseline (weight-level equivalent): centroid=857 Hz, RMS=0.100

ACT_DISCOVERIES = [
    # (layer, transform_fn, measured, description)
    ('input_bottleneck', roll_fn(32),
     'Δcent=−26%, ΔRMS=+37%, ΔZCR=−34%',
     'Cyclic channel rotation: rolling the 512-dim bottleneck output by 32 positions '
     'permutes which latent features are presented to the GRU per frame. '
     'Produces a dark, warm, louder timbre — comparable to weight roll but transient (per-call).'),
    ('input_bottleneck', flip_fn(),
     'Δcent=−23%, ΔRMS=+56%, ΔZCR=−30%',
     'Channel reversal: flipping the bottleneck feature order creates a similar darkening '
     'to roll but with mirrored feature relationships. Smooth waveform, louder output.'),
    ('gru', roll_fn(32),
     'Δcent=−12%, flatness=+5×, ΔRMS=−74%, ΔZCR=+1%',
     'GRU state permutation: rolling the GRU output displaces the coupling between '
     'hidden-state dimensions and the MLP input, attenuating output by −74% while '
     'introducing mild inharmonicity. The model generates a quieter, slightly distorted version.'),
    ('gru', noise_fn(2.0),
     'Δcent=−2%, flatness=+8×, ΔRMS=−45%, ΔZCR=+2%',
     'Recurrent state corruption: injecting large noise into the GRU output disrupts '
     'temporal coherence without dramatically shifting centroid. The model becomes '
     'quieter and slightly inharmonic — the GRU\'s temporal integration is degraded.'),
    ('inter_mlp', roll_fn(32),
     'Δcent=+490%, flatness=0.022, ΔRMS=+1183%, ΔZCR=+526%',
     'MLP output scramble: rolling the inter_mlp activations breaks all learned '
     'feature→synthesis-parameter associations. Produces wideband noise, much brighter '
     'and louder than baseline. The synthesis head receives mismatched features.'),
    ('output_params', roll_fn(32),
     'Δcent=−53%, ΔRMS=+2356%, ΔZCR=−41%',
     'Synthesis parameter shift: rolling the raw output_params activations (before sigmoid) '
     'by 32 cycles the assignment of synthesis parameters. The result is extremely loud '
     'and very dark — partial amplitudes and noise bands are mapped to wrong oscillators.'),
]

print('Activation-bending discoveries (probe latent, plaud-melody)\n')
print('BASELINE:')
display(Audio(decode(m_ref, z_probe).squeeze().numpy(), rate=SR))
print()

for layer, fn, measured, desc in ACT_DISCOVERIES:
    audio = bent_decode(m_ref, z_probe, layer, fn)
    wav = audio.squeeze().numpy()
    peak = np.abs(wav).max()
    wav_norm = np.clip(wav / (peak + 1e-6), -1, 1)
    print(f'── layer={layer}  {fn.__name__ if hasattr(fn, "__name__") else type(fn).__name__}')
    print(f'   Measured: {measured}')
    print(f'   {desc}')
    display(Audio(wav_norm, rate=SR))
    print()


In [ ]:
# ── Layer depth comparison: early vs late ────────────────────────────────
# Same transform (scale ×2), applied at each of the four interception points.
# Demonstrates how depth affects the perceptual character of bending.

fig, axes = plt.subplots(1, len(DECODE_LAYERS) + 1, figsize=(18, 3))

def specshow(ax, wav, title):
    n = (len(wav) // 512) * 512   # ensure length is divisible by frame size
    spec = np.abs(np.fft.rfft(wav[:n].reshape(-1, 512), axis=-1))
    ax.imshow(np.log1p(spec[:200].T), aspect='auto', origin='lower',
              cmap='inferno', interpolation='nearest')
    ax.set_title(title, fontsize=8)
    ax.set_xlabel('frame', fontsize=7)
    ax.set_ylabel('freq bin', fontsize=7)

# Baseline
audio_base = decode(m_ref, z_probe)
specshow(axes[0], audio_base.squeeze().numpy(), 'baseline')

# Each layer with scale ×2
for ax, layer in zip(axes[1:], DECODE_LAYERS):
    audio_bent = bent_decode(m_ref, z_probe, layer, scale_fn(2.0))
    wav = audio_bent.squeeze().numpy()
    peak = np.abs(wav).max()
    wav = wav / (peak + 1e-6)  # normalize for display
    specshow(ax, wav, f'scale×2 @{layer.split(".")[0][:12]}')

fig.suptitle('Scale×2 applied at different decode layers — spectrograms', fontsize=10)
plt.tight_layout()
plt.show()

print('Early bending (input_bottleneck): structural / timbral divergence')
print('Late bending (output_params): direct spectral reshaping, less deconstruction')

In [ ]:
# ── Interactive activation bending panel ──────────────────────────────────

act_layer_dd = widgets.Dropdown(
    options=DECODE_LAYERS, value='gru',
    description='layer:', style={'description_width': '50px'},
    layout=widgets.Layout(width='40%'),
)
act_transform_dd = widgets.Dropdown(
    options=['scale', 'shift', 'noise', 'roll', 'flip', 'channel shuffle', 'zero'],
    value='scale',
    description='transform:', style={'description_width': '80px'},
    layout=widgets.Layout(width='40%'),
)
act_strength = widgets.FloatSlider(
    value=1.5, min=-5.0, max=10.0, step=0.1,
    description='strength', continuous_update=False,
    layout=widgets.Layout(width='50%'),
    style={'description_width': '70px'},
)
act_model_dd = widgets.Dropdown(
    options=list(models.keys()), value='plaud-melody',
    description='model:', style={'description_width': '50px'},
    layout=widgets.Layout(width='40%'),
)
btn_act = widgets.Button(description='Bend & Listen', button_style='danger')
out_act = Output()

def on_act_bend(b):
    with out_act:
        out_act.clear_output(wait=True)
        m = models[act_model_dd.value]
        layer = act_layer_dd.value
        strength = act_strength.value
        t_name = act_transform_dd.value
        if t_name == 'scale':          fn = scale_fn(strength)
        elif t_name == 'shift':        fn = shift_fn(strength)
        elif t_name == 'noise':        fn = noise_fn(abs(strength) * 0.3)
        elif t_name == 'roll':         fn = roll_fn(int(strength * 50))
        elif t_name == 'flip':         fn = flip_fn()
        elif t_name == 'channel shuffle': fn = channel_shuffle_fn()
        else:                          fn = lambda x: torch.zeros_like(x)
        audio = bent_decode(m, z_probe, layer, fn)
        wav = audio.squeeze().numpy()
        peak = np.abs(wav).max()
        wav = wav / (peak + 1e-6)
        print(f'Bending: {t_name} (strength={strength:.2f}) @ {layer} on {act_model_dd.value}')
        display(Audio(np.clip(wav, -1, 1), rate=SR))

btn_act.on_click(on_act_bend)
display(widgets.VBox([
    widgets.HBox([act_model_dd, act_layer_dd, act_transform_dd]),
    act_strength,
    btn_act, out_act
]))

---
## 4.2  Weight-Level Network Bending (Model Rewriting)

Instead of intercepting activations *during* inference, we rewrite the model's **weights** before inference. The modified model then generates audio that reflects the new weights for every frame, without any per-call overhead.

**Modifying TorchScript parameters.** Direct assignment (`param = new_tensor`) doesn't work on scripted modules. The correct approach is `param.data.copy_(new_value)`. This is implemented in `src/network_bending/weights.py`.

**Always work on a fresh copy.** We reload from disk for each experiment so the original checkpoint stays clean across cells.

### Discoveries

The following transformations were identified empirically by running a systematic sweep across all decoder parameters with multiple operations (scale, shift, roll, flip, noise, zero), then measuring spectral features (centroid, spectral flatness, rolloff, ZCR, RMS) of the output audio and comparing to baseline.

Baseline (plaud-melody + LFO probe): centroid=857 Hz, flatness≈9.5×10⁻⁶ (highly tonal), RMS=0.100, ZCR=0.040

| Layer | Operation | Centroid | Flatness | RMS | ZCR | Character |
|-------|-----------|----------|----------|-----|-----|-----------|
| `input_bottleneck.6.weight` | roll(dim=0, shifts=10) | −29% | unchanged | +70% | −29% | Spectral darkening + warmth |
| `input_bottleneck.7.weight` | zero | −34% | unchanged | +89% | −45% | Deep bass emphasis |
| `gru.weight_ih_l0` | shift(−0.5) | −32% | unchanged | +73% | −35% | Fundamental focus, smooth |
| `gru.weight_ih_l0` | zero | −33% | unchanged | +108% | −44% | Input ablation — pure drone |
| `gru.weight_hh_l0` | shift(−0.5) | −31% | unchanged | −23% | −33% | Dark compression |
| `gru.weight_hh_l0` | shift(+0.5) | +38% | unchanged | −59% | +35% | Spectral brightening |
| `gru.weight_ih_l1` | scale(0.1) | −14% | unchanged | +37% | −19% | Mild warmth, more sustain |
| `inter_mlp.6.weight` | roll(dim=0, shifts=10) | +443% | 0.013 | +484% | +439% | Noise explosion |
| `inter_mlp.3.weight` | roll(dim=0, shifts=10) | +14% | 0.00017 | −91% | +19% | Near-silence with faint residual |

**Layer depth pattern:**  
- `input_bottleneck` roll → tonal but dark (energy shifts to fundamentals)  
- GRU weight shift (neg) → strong spectral darkening without adding noise  
- GRU weight zero → removes all input influence, leaving only recurrent dynamics  
- `inter_mlp.6` roll → completely breaks the feature→synthesis mapping, explosive noise  
- `inter_mlp.3` roll → disrupts the middle of the feature pathway, near-silence

In [ ]:

# ── Empirically discovered weight-bending transformations ─────────────────
# Each transformation was identified by sweeping operations across all
# decoder parameters and measuring: spectral centroid, flatness, RMS, ZCR.
# Descriptions below are derived from those measurements.

DISCOVERED_BENDS = [
    # (param_name, operation, kwargs, measured_effect, centroid_delta, description)
    (
        'pretrained.decoder.input_bottleneck.6.weight',
        'roll', dict(shifts=10, dim=0),
        'Δcent=−29%, ΔRMS=+70%, ΔZCR=−29%',
        'Spectral darkening + warmth: rolling the latent→hidden projection shifts '
        'energy toward lower harmonics. The waveform becomes smoother (lower ZCR) '
        'and louder. Timbre is warmer and more fundamental-dominant.',
    ),
    (
        'pretrained.decoder.input_bottleneck.7.weight',
        'zero_out', {},
        'Δcent=−34%, ΔRMS=+89%, ΔZCR=−45%',
        'Deep bass emphasis: zeroing the LayerNorm scale collapses all channel '
        'normalisation before the GRU. The largest centroid drop of any bottleneck '
        'operation, with the smoothest waveform and loudest output.',
    ),
    (
        'pretrained.decoder.gru.weight_ih_l0',
        'shift', dict(offset=-0.5),
        'Δcent=−32%, ΔRMS=+73%, ΔZCR=−35%',
        'Fundamental focus: negative-shifting the GRU input-to-hidden weights biases '
        'all gate activations toward inhibition, suppressing upper harmonics. '
        'Output is louder, darker and smoother — still tonal (flatness unchanged).',
    ),
    (
        'pretrained.decoder.gru.weight_ih_l0',
        'zero_out', {},
        'Δcent=−33%, ΔRMS=+108%, ΔZCR=−44%',
        'Input ablation: with zero input weights the GRU runs on its recurrent '
        'dynamics alone. The model collapses to a pure, very dark sustained tone — '
        'the loudest and smoothest output of any GRU weight operation.',
    ),
    (
        'pretrained.decoder.gru.weight_hh_l0',
        'shift', dict(offset=-0.5),
        'Δcent=−31%, ΔRMS=−23%, ΔZCR=−33%',
        'Dark compression: negative shift on the hidden-to-hidden weights dampens '
        'recurrent self-excitation. Output is darker AND quieter — the only GRU '
        'operation that simultaneously darkens and compresses.',
    ),
    (
        'pretrained.decoder.gru.weight_hh_l0',
        'shift', dict(offset=0.5),
        'Δcent=+38%, ΔRMS=−59%, ΔZCR=+35%',
        'Spectral brightening: positive shift has the opposing effect — upper '
        'harmonics are amplified while overall level drops. More buzzy, sibilant '
        'quality. Opposite polarity from shift(−0.5).',
    ),
    (
        'pretrained.decoder.gru.weight_ih_l1',
        'scale', dict(factor=0.1),
        'Δcent=−14%, ΔRMS=+37%, ΔZCR=−19%',
        'Second-layer GRU contraction: scaling down the second-layer input weights '
        'reduces l1\'s contribution, gently warming the timbre. Less dramatic than '
        'l0 operations — l1 acts on already-processed recurrent state.',
    ),
    (
        'pretrained.decoder.inter_mlp.3.weight',
        'roll', dict(shifts=10, dim=0),
        'Δcent=+14%, flatness=0.00017, ΔRMS=−91%, ΔZCR=+19%',
        'Near-silence / faint residual: disrupting the second MLP layer drastically '
        'attenuates output (−91% RMS). A faint, slightly inharmonic trace remains. '
        'This layer is critical for the model\'s ability to pass signal through.',
    ),
    (
        'pretrained.decoder.inter_mlp.6.weight',
        'roll', dict(shifts=10, dim=0),
        'Δcent=+443%, flatness=0.013, ΔRMS=+484%, ΔZCR=+439%',
        'Noise explosion: rolling the penultimate MLP layer disconnects learned '
        'feature→synthesis-parameter mapping entirely. Output becomes wideband noise '
        'at 5× the baseline brightness and 5× louder. The most dramatic single-layer '
        'weight operation in the model.',
    ),
]

MODEL_NAME = 'plaud-melody'
print(f'Weight-bending discoveries on {MODEL_NAME}')
print(f'Baseline: centroid≈857 Hz, flatness≈9.5e-6 (highly tonal), RMS=0.100\n')
print('BASELINE:')
display(Audio(decode(models[MODEL_NAME], z_probe).squeeze().numpy(), rate=SR))
print()

for param_name, op, kwargs, measured, desc in DISCOVERED_BENDS:
    m_fresh = torch.jit.load(model_paths[MODEL_NAME], map_location='cpu').eval()
    getattr(wb, op)(m_fresh, param_name, **kwargs)
    audio = decode(m_fresh, z_probe)
    wav = audio.squeeze().numpy()
    peak = np.abs(wav).max()
    wav_norm = np.clip(wav / (peak + 1e-6), -1, 1)
    layer_short = param_name.split('pretrained.decoder.')[1]
    print(f'──  {layer_short}  |  {op}({", ".join(f"{k}={v}" for k,v in kwargs.items())})')
    print(f'    Measured: {measured}')
    print(f'    {desc}')
    display(Audio(wav_norm, rate=SR))
    print()


In [ ]:
# ── Interactive weight bending panel ──────────────────────────────────────

wb_model_dd = widgets.Dropdown(
    options=list(models.keys()), value='plaud-melody',
    description='model:', style={'description_width': '50px'},
    layout=widgets.Layout(width='40%'),
)

# Only expose decoder parameters (encoder params have little effect on decode path)
decoder_params = [n for n, _ in models['plaud-melody'].named_parameters()
                  if 'decoder' in n]
wb_param_dd = widgets.Dropdown(
    options=decoder_params,
    value='pretrained.decoder.gru.weight_ih_l0',
    description='param:', style={'description_width': '50px'},
    layout=widgets.Layout(width='80%'),
)
wb_op_dd = widgets.Dropdown(
    options=['scale', 'shift', 'roll', 'flip', 'noise', 'zero'],
    value='scale',
    description='operation:', style={'description_width': '80px'},
    layout=widgets.Layout(width='40%'),
)
wb_strength = widgets.FloatSlider(
    value=2.0, min=-5.0, max=10.0, step=0.05,
    description='strength', continuous_update=False,
    layout=widgets.Layout(width='60%'),
    style={'description_width': '70px'},
)
btn_wb = widgets.Button(description='Bend & Listen', button_style='warning')
out_wb = Output()

def on_wb_bend(b):
    with out_wb:
        out_wb.clear_output(wait=True)
        m_fresh = torch.jit.load(model_paths[wb_model_dd.value], map_location='cpu').eval()
        op = wb_op_dd.value
        param = wb_param_dd.value
        s = wb_strength.value
        if op == 'scale':    wb.scale(m_fresh, param, s)
        elif op == 'shift':  wb.shift(m_fresh, param, s)
        elif op == 'roll':   wb.roll(m_fresh, param, int(s * 20))
        elif op == 'flip':   wb.flip(m_fresh, param)
        elif op == 'noise':  wb.add_noise(m_fresh, param, abs(s) * 0.1)
        elif op == 'zero':   wb.zero_out(m_fresh, param)
        audio = decode(m_fresh, z_probe)
        wav = audio.squeeze().numpy()
        peak = np.abs(wav).max()
        wav = wav / (peak + 1e-6)
        print(f'{op} on {param.split("pretrained.decoder.")[1]}')
        display(Audio(np.clip(wav, -1, 1), rate=SR))

btn_wb.on_click(on_wb_bend)
display(widgets.VBox([
    widgets.HBox([wb_model_dd, wb_op_dd]),
    wb_param_dd,
    wb_strength,
    btn_wb, out_wb
]))

---
## 4.3  Model Blending

Linear interpolation of weights between two PLAUD checkpoints:

$$W_{\text{blend}} = \alpha \cdot W_A + (1-\alpha) \cdot W_B$$

At α=1 you get model A; at α=0 you get model B. In between, you explore the *interpolant* — a model that has never been trained but whose weights form a meaningful path between two trained points.

### Blending strategy (from §4.0 compatibility analysis)

Three pairs are **fully compatible** — every parameter shape matches:
- `plaud-melody` ↔ `tarta-lows-avg`
- `plaud-melody` ↔ `iclc-guitar-loops`  
- `tarta-lows-avg` ↔ `iclc-guitar-loops`

`plaud-drums` is **partially compatible** with all others (48/50 params match; `output_params.weight` and `.bias` differ: 1065×512 vs 665×512). For drums pairs we use **partial blending**: match-shaped parameters are interpolated, the drums `output_params` falls back to model A.

BatchNorm: PLAUD uses **LayerNorm** (not BatchNorm), so there are no `running_mean`/`running_var` buffers to handle.

In [ ]:
# ── Full blend: plaud-melody ↔ tarta-lows-avg ─────────────────────────────
# Fully compatible pair — all 50 parameters are interpolated.

BLEND_A_NAME = 'plaud-melody'
BLEND_B_NAME = 'tarta-lows-avg'
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]

m_A = models[BLEND_A_NAME]
m_B = models[BLEND_B_NAME]

ck = compatible_keys(m_A, m_B)
print(f'{BLEND_A_NAME} ↔ {BLEND_B_NAME}: {len(ck)}/{len(list(m_A.parameters()))} parameters blendable')
print(f'Strategy: FULL blend (all parameters interpolated)\n')

print('α = 0.0  →  pure', BLEND_B_NAME)
print('α = 1.0  →  pure', BLEND_A_NAME)
print()

blended_audio = []
for alpha in ALPHAS:
    m_blend = bld.blend(m_A, m_B, alpha, model_paths[BLEND_A_NAME])
    audio = decode(m_blend, z_probe)
    wav = audio.squeeze().numpy()
    peak = np.abs(wav).max()
    wav_norm = wav / (peak + 1e-6)
    blended_audio.append(wav_norm)
    print(f'α={alpha:.2f}  peak={peak:.3f}')
    display(Audio(np.clip(wav_norm, -1, 1), rate=SR))

In [ ]:
# ── Spectrogram strip across the alpha sweep ──────────────────────────────

fig, axes = plt.subplots(1, len(ALPHAS), figsize=(18, 3), sharey=True)
for ax, alpha, wav in zip(axes, ALPHAS, blended_audio):
    # Short-time Fourier transform
    N = 512
    hop_spec = 128
    n_frames_spec = (len(wav) - N) // hop_spec
    spec = np.zeros((N//2+1, n_frames_spec))
    win = np.hanning(N)
    for i in range(n_frames_spec):
        seg = wav[i*hop_spec : i*hop_spec + N] * win
        spec[:, i] = np.abs(np.fft.rfft(seg))
    ax.imshow(np.log1p(spec[:128]), aspect='auto', origin='lower',
              cmap='inferno', interpolation='nearest')
    ax.set_title(f'α={alpha:.2f}', fontsize=9)
    ax.set_xlabel('frame', fontsize=7)
    if ax == axes[0]:
        ax.set_ylabel('freq bin', fontsize=7)

fig.suptitle(f'Blend sweep: {BLEND_A_NAME} (α=1) → {BLEND_B_NAME} (α=0)', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Partial blend: plaud-drums ↔ plaud-melody ─────────────────────────────
# 48/50 parameters match; output_params (665 vs 1065) does not.
# Blendable fraction: GRU, inter_mlp, input_bottleneck, encoder — all the
# recurrent and projection layers. output_params falls back to model A (drums).

PARTIAL_A = 'tarta-lows-avg'
PARTIAL_B = 'plaud-melody'

m_PA = models[PARTIAL_A]
m_PB = models[PARTIAL_B]
ck_partial = compatible_keys(m_PA, m_PB)

mismatch = mismatched_params(m_PA, m_PB)
print(f'{PARTIAL_A} ↔ {PARTIAL_B}: {len(ck_partial)}/50 parameters blendable')
print('Non-blendable (shape mismatch):')
for name, sa, sb in mismatch:
    print(f'  {name}: {sa} vs {sb}')
print(f'Fallback: non-blendable params take values from model A ({PARTIAL_A})\n')

ALPHAS_PARTIAL = [0.0, 0.33, 0.66, 1.0]
for alpha in ALPHAS_PARTIAL:
    m_blend = bld.blend(m_PA, m_PB, alpha, model_paths[PARTIAL_A], fallback='a')
    audio = decode(m_blend, z_probe)
    wav = audio.squeeze().numpy()
    peak = np.abs(wav).max()
    wav_norm = wav / (peak + 1e-6)
    print(f'α={alpha:.2f}  peak={peak:.3f}')
    display(Audio(np.clip(wav_norm, -1, 1), rate=SR))

In [ ]:
# ── Interactive blend panel ────────────────────────────────────────────────

bl_a_dd = widgets.Dropdown(
    options=list(models.keys()), value='plaud-melody',
    description='Model A:', style={'description_width': '65px'},
    layout=widgets.Layout(width='45%'),
)
bl_b_dd = widgets.Dropdown(
    options=list(models.keys()), value='tarta-lows-avg',
    description='Model B:', style={'description_width': '65px'},
    layout=widgets.Layout(width='45%'),
)
bl_alpha = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description='α (→A)', continuous_update=False,
    layout=widgets.Layout(width='60%'),
    style={'description_width': '70px'},
)
bl_compat_label = widgets.Label(value='')
btn_bl = widgets.Button(description='Blend & Listen', button_style='success')
out_bl = Output()

def update_compat_label(*args):
    if bl_a_dd.value == bl_b_dd.value:
        bl_compat_label.value = 'Same model — trivial blend'
        return
    ck = compatible_keys(models[bl_a_dd.value], models[bl_b_dd.value])
    n_total = len(list(models[bl_a_dd.value].parameters()))
    mode = 'FULL' if len(ck) == n_total else f'PARTIAL ({len(ck)}/{n_total})'
    bl_compat_label.value = f'Compatibility: {mode}'

bl_a_dd.observe(update_compat_label, 'value')
bl_b_dd.observe(update_compat_label, 'value')
update_compat_label()

def on_blend(b):
    with out_bl:
        out_bl.clear_output(wait=True)
        name_a, name_b = bl_a_dd.value, bl_b_dd.value
        alpha = bl_alpha.value
        m_blend = bld.blend(models[name_a], models[name_b], alpha, model_paths[name_a])
        audio = decode(m_blend, z_probe)
        wav = audio.squeeze().numpy()
        peak = np.abs(wav).max()
        wav = wav / (peak + 1e-6)
        ck = compatible_keys(models[name_a], models[name_b])
        n_total = len(list(models[name_a].parameters()))
        print(f'α={alpha:.2f}: {alpha:.0%} {name_a} + {1-alpha:.0%} {name_b}')
        print(f'Blended {len(ck)}/{n_total} parameters')
        display(Audio(np.clip(wav, -1, 1), rate=SR))

btn_bl.on_click(on_blend)
display(widgets.VBox([
    widgets.HBox([bl_a_dd, bl_b_dd]),
    bl_compat_label,
    bl_alpha,
    btn_bl, out_bl
]))